# Exercise 65 — Loading a CSV into SQLite

## Concept

A classic DE chore: take a CSV someone gave you and make it queryable in a database. The pattern:

1. Inspect the CSV (columns, types you'll need).
2. Create the target table with explicit column types.
3. Stream the CSV row by row, inserting via `executemany` in batches.

```python
import csv, sqlite3

conn = sqlite3.connect("app.db")
conn.execute("""
    CREATE TABLE IF NOT EXISTS orders (
        order_id INTEGER PRIMARY KEY,
        customer TEXT,
        amount   REAL,
        paid     INTEGER             -- SQLite uses 0/1 for booleans
    )
""")

with open("orders.csv", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    batch = []
    for row in reader:
        batch.append((
            int(row["order_id"]),
            row["customer"],
            float(row["amount"]),
            1 if row["paid"].lower() == "true" else 0,
        ))
        if len(batch) >= 1000:
            conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", batch)
            batch.clear()
    if batch:
        conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", batch)

conn.commit()
conn.close()
```

### Why batching matters

A loop of `execute()` calls is one round-trip per row. `executemany()` with batches of 500–5000 amortizes the overhead. For a few million rows, batching can be 50× faster than per-row inserts.

### Type conversion is YOUR job

`csv.DictReader` returns strings. SQLite is loose with types, but you should still cast — otherwise `amount` ends up stored as a string and `WHERE amount > 100` does string comparison.

## Setup

In [ ]:
from pathlib import Path

Path("orders.csv").write_text(
    "order_id,customer,amount,paid\n"
    "1,Alice,120.50,true\n"
    "2,Bob,45.00,true\n"
    "3,Alice,300.75,false\n"
    "4,Carol,89.99,true\n"
    "5,Bob,210.00,true\n"
    "6,Alice,55.25,false\n"
    "7,Carol,430.10,true\n",
    encoding="utf-8",
)
print("orders.csv ready")


## Task 1 — Schema-aware load

Write a function that:
1. Creates table `orders(order_id INTEGER PRIMARY KEY, customer TEXT, amount REAL, paid INTEGER)` if missing.
2. Loads rows from the given CSV, casting each value to the correct Python type.
3. Returns the number of rows inserted.

```python
def load_orders(csv_path: str, db_path: str) -> int:
    ...
```

Use `executemany` (build a list of tuples), one commit at the end. Open the connection inside the function and close it before returning.

In [ ]:
import csv
import sqlite3
from pathlib import Path

DB = "orders.db"
Path(DB).unlink(missing_ok=True)

def load_orders(csv_path: str, db_path: str) -> int:
    pass  # TODO

n = load_orders("orders.csv", DB)
assert n == 7

with sqlite3.connect(DB) as conn:
    count = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
    types = conn.execute("SELECT typeof(amount), typeof(paid) FROM orders LIMIT 1").fetchone()
assert count == 7
assert types == ("real", "integer")
print("ok")


## Task 2 — Query the loaded table

Write a function that returns the **total amount of paid orders per customer**, as a list of `(customer, total)` tuples sorted by total descending.

```python
def paid_totals_by_customer(db_path: str) -> list[tuple[str, float]]:
    ...
```

Do this with a single SQL `GROUP BY` query. Don't pull all rows into Python and aggregate — let SQL do its job.

Expected (paid=1 rows only):
```
[("Carol", 520.09), ("Bob", 255.0), ("Alice", 120.5)]
```

In [ ]:
import sqlite3

def paid_totals_by_customer(db_path: str) -> list[tuple[str, float]]:
    pass  # TODO

result = paid_totals_by_customer(DB)
assert result == [("Carol", 520.09), ("Bob", 255.0), ("Alice", 120.5)]
print("ok")


## Task 3 — Batched load for large files

Rewrite the load so it commits in **batches** of `batch_size` rows. This matters when the CSV is millions of rows — committing per row is slow, and a single commit at the end means everything must fit in one transaction.

```python
def load_orders_batched(csv_path: str, db_path: str, batch_size: int) -> int:
    """Insert rows in batches of `batch_size`, committing after each batch."""
```

Reset the DB first (drop and recreate the table). Verify the row count after.

In [ ]:
import csv
import sqlite3
from pathlib import Path

DB2 = "orders_batched.db"
Path(DB2).unlink(missing_ok=True)

def load_orders_batched(csv_path: str, db_path: str, batch_size: int) -> int:
    pass  # TODO

n = load_orders_batched("orders.csv", DB2, batch_size=3)
assert n == 7
with sqlite3.connect(DB2) as conn:
    count = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
assert count == 7
print("ok")
